# 2-2-3 缺失值處理策略

本單元比較三種處理缺失值（NaN）的策略：

1. **刪除法**：`dropna()`，簡單但會損失資料
2. **統計補值**：用全體平均數 `fillna()`，會抹平群體差異
3. **商業邏輯補值**：`groupby().transform()` 依群組各自的平均補值，最精準

## 1. 載入資料

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('E-commerce_Dirty_Dataset.csv')
df.info()

## 2. 備份三份資料，分別實驗三種策略

先用 `copy()` 備份，避免策略之間互相污染。
同時統一 `user_type` 的大小寫（沿用 2-1-3 的成果），確保分組正確。

In [ ]:
df_strategy1 = df.copy()
df_strategy2 = df.copy()
df_strategy3 = df.copy()

for d in [df_strategy1, df_strategy2, df_strategy3]:
    d['user_type'] = d['user_type'].str.upper().replace({'VIP': 'VIP', 'REGULAR': 'Regular', 'LOW': 'Low'})

df_strategy1['user_type'].value_counts()

## 策略一：刪除法 (Deletion)

- `dropna(subset=[...])`：只針對核心欄位，有缺失就剔除
- `dropna()`：任何欄位有缺失就整列刪除（一刀切，常常刪掉大量資料）

In [ ]:
print(f"處理前資料總筆數: {len(df_strategy1)} 筆")

df_strategy1_cleaned = df_strategy1.dropna(subset=['spending'])
print(f"僅刪除 spending 缺失後筆數: {len(df_strategy1_cleaned)} 筆")

df_strategy1_extreme = df_strategy1.dropna()
print(f"全欄位一刀切 dropna() 後殘存筆數: {len(df_strategy1_extreme)} 筆")

## 策略二：統計補值 (Statistical Imputation)

用「全站平均數」填補 `income`。先用 `pd.to_numeric(errors='coerce')` 把無法轉換的值變成 NaN。

> 缺點：不論 VIP 或 Low，都被補成同一個數字，會扭曲各群體的真實面貌。

In [ ]:
df_strategy2['income'] = pd.to_numeric(df_strategy2['income'], errors='coerce')
print(f"income 的缺失情形: {df_strategy2['income'].isnull().sum()} 筆")

original_income_mean = df_strategy2['income'].mean()
print(f"補值前平均收入: {original_income_mean:.2f} 元")

In [ ]:
df_strategy2['income_filled_global'] = df_strategy2['income'].fillna(original_income_mean)

filled_income_mean = df_strategy2['income_filled_global'].mean()
print(f"補值後平均收入: {filled_income_mean:.2f} 元")

### 自行練習：改用中位數

中位數較不受極端值影響，可以試試看（取消註解後執行）：

In [ ]:
# original_income_median = df_strategy2['income'].median()
# df_strategy2['income_filled_global'] = df_strategy2['income'].fillna(original_income_median)

## 策略三：商業邏輯補值 (Business Logic Imputation)

VIP、Regular、Low 的收入差異很大，所以應依 `user_type` 分組，用「各自群組的平均」補值。

`groupby('user_type')['income'].transform('mean')` 會把「群組平均值」對應回**每一列**，
因此能直接餵給 `fillna()`，讓每筆缺失值自動對號入座。

In [ ]:
df_strategy3['income'] = pd.to_numeric(df_strategy3['income'], errors='coerce')

# 先單獨看 transform 的結果：每一列都帶有其群組的平均收入
df_strategy3.groupby('user_type')['income'].transform('mean').head()

In [ ]:
df_strategy3['income_filled_logic'] = df_strategy3['income'].fillna(
    df_strategy3.groupby('user_type')['income'].transform('mean')
)
df_strategy3[['user_type', 'income', 'income_filled_logic']].head(10)

## 商務回測：兩種補值策略如何改變數據？

先看 Low 用戶被補值後的平均收入。

In [ ]:
low_compare = pd.DataFrame({
    '策略二：直接使用平均值': [df_strategy2[df_strategy2['user_type'] == 'Low']['income_filled_global'].mean()],
    '策略三：商業邏輯補值': [df_strategy3[df_strategy3['user_type'] == 'Low']['income_filled_logic'].mean()]
}, index=['Low 用戶補值後的平均收入'])

low_compare

## 全客群策略效果對照

1. **基準點**：排除缺失值與負數後，各組的真實平均收入（控制組）
2. **策略二**：補值後各組平均
3. **策略三**：補值後各組平均

比較三者與基準點的差距，即可看出哪種策略更貼近真實。

In [ ]:
df_clean_baseline = df.copy()
df_clean_baseline['user_type'] = df_clean_baseline['user_type'].str.upper().replace({'VIP': 'VIP', 'REGULAR': 'Regular', 'LOW': 'Low'})
df_clean_baseline['income'] = pd.to_numeric(df_clean_baseline['income'], errors='coerce')
df_clean_baseline = df_clean_baseline[df_clean_baseline['income'] > 0]  # 排除負數
baseline_income = df_clean_baseline.groupby('user_type')['income'].mean()

strategy2_income = df_strategy2.groupby('user_type')['income_filled_global'].mean()
strategy3_income = df_strategy3.groupby('user_type')['income_filled_logic'].mean()

full_compare = pd.DataFrame({
    '基準點：未補值的真實均值 (元)': baseline_income,
    '策略二：直接使用全站平均 (元)': strategy2_income,
    '策略三：商業邏輯分組補值 (元)': strategy3_income
})

full_compare.reindex(['VIP', 'Regular', 'Low']).round(2)